In [94]:
import pandas as pd
import re
import numpy as np

pd.set_option('display.max_rows', None)

In [95]:
df = pd.read_excel('lul.xlsx')

In [96]:
df.rename(columns={'Unnamed: 1': 'date', 'Unnamed: 2': 'event'}, inplace=True)

In [97]:
df.drop(columns=['Unnamed: 0'], inplace=True)

In [98]:
df['date'] = df['date'].astype(str).str.replace(r'\s+00:00:00$', '', regex=True).replace('nan', np.nan)

In [99]:
def parse_flexible_date(text):
    if pd.isna(text) or str(text).lower() == 'nan':
        return pd.Series([np.nan] * 8) # 8 columns now
    
    original_text = str(text).lower()
    
    # 1. Configuration
    seasons_list = ['tavasz', 'nyár', 'ősz', 'tél']
    months_map = {
        'január': '01', 'február': '02', 'március': '03', 'április': '04',
        'május': '05', 'június': '06', 'július': '07', 'augusztus': '08', 
        'szeptember': '09', 'október': '10', 'november': '11', 'december': '12'
    }

    # 2. Split by range indicators
    parts = re.split(r'\s*[-–—]\s*', original_text)
    
    is_range = False
    if len(parts) == 2:
        is_range = True
    elif len(parts) > 2:
        if re.search(r'\d{4}', parts[-1]):
            is_range = True
        else:
            is_range = False

    def extract_from_segment(segment):
        if not segment or not segment.strip():
            return np.nan, np.nan, np.nan, np.nan
        
        clean_seg = segment.replace('.', ' ').replace('/', ' ')
        y_match = re.search(r'(\d{4})', clean_seg)
        y = y_match.group(1) if y_match else None
        
        # Identify Season (Text-based)
        res_season = np.nan
        for s in seasons_list:
            if s in segment:
                res_season = s
                break
        
        # Identify Named Months
        found_m = []
        for name, val in months_map.items():
            if name in segment:
                found_m.append(val)
        
        # Extract 1-2 digit numbers (1-31)
        nums = re.findall(r'(?<!\d)([0-3]?\d)(?!\d)', clean_seg)
        y_suffix = str(y)[-2:] if y is not None else "____"
        
        valid_nums = [
            n.zfill(2) for n in nums 
            if 1 <= int(n) <= 31 and n.zfill(2) != y_suffix and n != y
        ]

        final_m = found_m.copy()
        final_d = []
        
        for num in valid_nums:
            # If we don't have a month from text and this number is 1-12
            if len(final_m) < 1 and int(num) <= 12:
                final_m.append(num)
            # Otherwise it's a day
            elif len(final_d) < 1:
                final_d.append(num)
        
        res_m = final_m[0] if len(final_m) > 0 else np.nan
        res_d = final_d[0] if len(final_d) > 0 else np.nan
        
        return (y if y is not None else np.nan), res_m, res_d, res_season

    # 3. Assign Start and End columns
    if is_range:
        s_y, s_m, s_d, s_s = extract_from_segment(parts[0])
        e_y, e_m, e_d, e_s = extract_from_segment("-".join(parts[1:]))
    else:
        s_y, s_m, s_d, s_s = extract_from_segment(original_text)
        e_y, e_m, e_d, e_s = np.nan, np.nan, np.nan, np.nan

    return pd.Series([s_y, s_m, s_d, s_s, e_y, e_m, e_d, e_s])

In [100]:
new_cols = [
    'startyear', 'startmonth', 'startday', 'startseason', 
    'endyear', 'endmonth', 'endday', 'endseason'
]
df[new_cols] = df['date'].apply(parse_flexible_date)

In [101]:
df.drop(columns=['date'], inplace=True)

In [102]:
season_sort_map = {
    'tavasz': '03', 
    'nyár': '06', 
    'ősz': '09', 
    'tél': '12'
}

def create_sort_key(row):
    # Get Year (Default to '0000' if NaN for sorting)
    y = str(row['startyear']) if pd.notna(row['startyear']) else '0000'
    
    # Get Month
    m = str(row['startmonth']) if pd.notna(row['startmonth']) else None
    
    # If Month is missing, check Season
    if not m or m == 'nan':
        s = str(row['startseason']).lower()
        m = season_sort_map.get(s, '01') # Default to Jan if no month or season
    
    # Get Day
    d = str(row['startday']) if pd.notna(row['startday']) else '01'
    
    # Create a string like '20231023' which sorts perfectly alphabetically
    return f"{y}{m.zfill(2)}{d.zfill(2)}"

# 2. Create the temporary sort key
df['temp_sort_key'] = df.apply(create_sort_key, axis=1)

# 3. Sort the DataFrame
# We sort by the key, and use the 'event' or 'endyear' as a tie-breaker
df = df.sort_values(by=['temp_sort_key', 'endyear']).drop(columns=['temp_sort_key'])

# Reset index to reflect the new chronological order
df = df.reset_index(drop=True)

In [103]:
df['event'] = df['event'].astype(str).apply(lambda x: x[0].upper() + x[1:] if len(x) > 0 else x)

In [104]:
df.to_csv('lul.csv', index=False, sep=';')